## Loading the session saved in sajjad_01_exploration

In [2]:
from pyspark.sql import SparkSession
from pathlib import Path

spark = SparkSession.builder \
    .appName("SessionAnalysis") \
    .master("local[*]") \
    .getOrCreate()

sessions_path = Path.cwd().parent / "output" / "sessions.parquet"

df = spark.read.parquet(str(sessions_path))
df.show(5)

26/04/04 10:52:28 WARN Utils: Your hostname, Sajjads-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 192.168.1.108 instead (on interface en0)
26/04/04 10:52:28 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/04 10:52:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/04 10:52:29 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


+----------+--------+------------------+------------+---------+------+-------+----------+
|session_id|n_clicks|         avg_price|n_categories|n_colours|bought|country|      date|
+----------+--------+------------------+------------+---------+------+-------+----------+
|       496|       2|              45.0|           1|        2|  true|      9|2008-04-02|
|      5803|       4|              35.5|           1|        4| false|     44|2008-04-24|
|      9900|       7|55.142857142857146|           1|        6|  true|      9|2008-05-19|
|      3794|       9|43.888888888888886|           3|        6|  true|     29|2008-04-15|
|      3997|      11| 39.36363636363637|           2|        7|  true|     29|2008-04-16|
+----------+--------+------------------+------------+---------+------+-------+----------+
only showing top 5 rows



### extracting the days and onth from the date column!

In [3]:
df = df.withColumn("month", F.month("date")) \
       .withColumn("day_of_week", F.dayofweek("date"))

df.select("session_id", "date", "month", "day_of_week").show(5)

+----------+----------+-----+-----------+
|session_id|      date|month|day_of_week|
+----------+----------+-----+-----------+
|       496|2008-04-02|    4|          4|
|      5803|2008-04-24|    4|          5|
|      9900|2008-05-19|    5|          2|
|      3794|2008-04-15|    4|          3|
|      3997|2008-04-16|    4|          4|
+----------+----------+-----+-----------+
only showing top 5 rows



### cast *bought* from boolean to integer so mlib can use it as a label

In [ ]:
df = df.withColumn("label", F.col("bought").cast("int")).drop("bought")
c
df.select("session_id", "label").show(5)

+----------+-----+
|session_id|label|
+----------+-----+
|       496|    1|
|      5803|    0|
|      9900|    1|
|      3794|    1|
|      3997|    1|
+----------+-----+
only showing top 5 rows



#### numeric columns => single feature vector.  
#### all ML algos in mlib require a singl feature column and it needs to be a vector

In [5]:
from pyspark.ml.feature import VectorAssembler, StandardScaler

feature_cols = ["n_clicks", "avg_price", "n_categories", "n_colours", "country", "month", "day_of_week"]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features_raw")
df = assembler.transform(df)
df.select("session_id", "features_raw").show(5, truncate=False)

+----------+---------------------------------------------+
|session_id|features_raw                                 |
+----------+---------------------------------------------+
|496       |[2.0,45.0,1.0,2.0,9.0,4.0,4.0]               |
|5803      |[4.0,35.5,1.0,4.0,44.0,4.0,5.0]              |
|9900      |[7.0,55.142857142857146,1.0,6.0,9.0,5.0,2.0] |
|3794      |[9.0,43.888888888888886,3.0,6.0,29.0,4.0,3.0]|
|3997      |[11.0,39.36363636363637,2.0,7.0,29.0,4.0,4.0]|
+----------+---------------------------------------------+
only showing top 5 rows



### Scaling the features

In [ ]:
scaler = StandardScaler(inputCol="features_raw", outputCol="features", withMean=True, withStd=True)
# with withMean=True : mean =0 / withStd=True : scaled by STD
scaler_model = scaler.fit(df)
df = scaler_model.transform(df).drop("features_raw")

df.select("session_id", "features", "label").show(5, truncate=False)

+----------+---------------------------------------------------------------------------------------------------------------------------------------------+-----+
|session_id|features                                                                                                                                     |label|
+----------+---------------------------------------------------------------------------------------------------------------------------------------------+-----+
|496       |[-0.5433242351532607,0.073933134711089,-0.8904123852598286,-0.598722404905169,-2.5977839721396965,-1.200664148035872,0.06302665484370627]    |1    |
|5803      |[-0.3209824580792501,-1.0496823671860451,-0.8904123852598286,0.16428106935631234,2.301818632333039,-1.200664148035872,0.6051957952930532]    |0    |
|9900      |[0.012530207531765918,1.2735827683155483,-0.8904123852598286,0.9272845436177938,-2.5977839721396965,-0.43918388457440066,-1.0213116260549877]|1    |
|3794      |[0.23487198460577657,-

In [ ]:
output_path = Path.cwd().parent / "output" / "features.parquet"
df.select("session_id", "features", "label").write.parquet(str(output_path), mode="overwrite")
print("these are now saved: session_id | features | label")